# Notebook 4: Utvärdering & Slutsatser

---

Detta är projektets avslutande notebook. Här **jämför vi alla modeller**,
visualiserar deras prestanda och besvarar projektets kärnfråga:

> *Kan modellen förbättras – och hur?*

Den här notebooken fungerar som **presentationsunderlag** för er 15-minutersdemo.

## Modelljämförelse

Vi laddar resultatfilen som skapades i Notebook 3 och sätter upp visualiseringar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.metrics import (
    RocCurveDisplay, ConfusionMatrixDisplay,
    roc_curve, auc, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from umap import UMAP
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style='whitegrid', palette='muted')

# Ladda processad data
X_train = pd.read_csv('Data/processed/X_train_scaled.csv').values
X_test  = pd.read_csv('Data/processed/X_test_scaled.csv').values
y_train = pd.read_csv('Data/processed/y_train.csv').values.ravel()
y_test  = pd.read_csv('Data/processed/y_test.csv').values.ravel()

# Ladda resultat från Notebook 3
df_resultat = pd.read_csv('outputs/models/results.csv', index_col=0)

print('Laddat resultat för', len(df_resultat), 'modeller.')
df_resultat[['Accuracy', 'F1-score', 'ROC-AUC', 'CV F1 (5-fold)']].round(4)

In [ ]:
# Stapeldiagram – jämför alla modellers Accuracy, F1 och AUC sida vid sida
matt = ['Accuracy', 'F1-score', 'ROC-AUC']
farger = ['steelblue', 'tomato', 'seagreen']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for i, matt_namn in enumerate(matt):
    ax = axes[i]
    df_resultat[matt_namn].sort_values().plot(
        kind='barh', ax=ax, color=farger[i], edgecolor='white')
    ax.set_title(matt_namn, fontsize=13, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_xlim(0, 1)
    for j, v in enumerate(df_resultat[matt_namn].sort_values()):
        ax.text(v + 0.005, j, f'{v:.3f}', va='center', fontsize=9)

plt.suptitle('Modelljämförelse – Alla modeller', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/04_modelljamforelse.png', dpi=150)
plt.show()

## ROC-kurvor

ROC-kurvan (Receiver Operating Characteristic) visar avvägningen mellan
**True Positive Rate** (TPR, andel sjuka som identifieras rätt)
och **False Positive Rate** (FPR, andel friska som felklassificeras som sjuka).

**AUC** (Area Under the Curve) sammanfattar ROC-kurvan i ett tal:
- AUC = 1.0 → perfekt modell
- AUC = 0.5 → slumpmässig gissning (diagonal linje)

Vi återskapar modellerna för att kunna rita ROC-kurvor.

In [ ]:
# Återskapa dimensionsreduktionstransformationer
basta_k = int(df_resultat.filter(like='KNN', axis=0)
              .filter(like='PCA', axis=0).index[0]
              .split('k=')[1].split(')')[0])

# PCA
n_komp_90 = int(df_resultat.filter(like='PCA', axis=0).index[0]
                .split('komp')[0].split('(')[1].strip())
pca = PCA(n_components=n_komp_90, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

# UMAP
umap_klass = UMAP(n_components=10, random_state=42, n_neighbors=30, min_dist=0.1)
X_train_umap = umap_klass.fit_transform(X_train)
X_test_umap  = umap_klass.transform(X_test)

# Definiera alla modeller och deras dataset
modell_config = [
    ('Logistisk Regression',    LogisticRegression(max_iter=1000, random_state=42),  X_train, X_test),
    (f'KNN (k={basta_k})',      KNeighborsClassifier(n_neighbors=basta_k),             X_train, X_test),
    ('Beslutsträd',             DecisionTreeClassifier(max_depth=5, random_state=42),  X_train, X_test),
    ('Random Forest',           RandomForestClassifier(n_estimators=200, random_state=42), X_train, X_test),
    (f'LR + PCA',               LogisticRegression(max_iter=1000, random_state=42),  X_train_pca, X_test_pca),
    (f'KNN + PCA',              KNeighborsClassifier(n_neighbors=basta_k),             X_train_pca, X_test_pca),
    ('LR + UMAP',               LogisticRegression(max_iter=1000, random_state=42),  X_train_umap, X_test_umap),
    ('KNN + UMAP',              KNeighborsClassifier(n_neighbors=basta_k),             X_train_umap, X_test_umap),
]

fig, ax = plt.subplots(figsize=(10, 8))
for namn, modell, Xtr, Xte in modell_config:
    modell.fit(Xtr, y_train)
    y_prob = modell.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{namn}  (AUC={roc_auc:.3f})', linewidth=1.5)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Slumpmässig (AUC=0.500)')
ax.set_title('ROC-kurvor – alla modeller', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/figures/04_roc_kurvor.png', dpi=150)
plt.show()

## Konfusionsmatriser

En konfusionsmatris visar exakt hur modellen klassificerade:

| | Förutsagt Frisk | Förutsagt Sjuk |
|---|---|---|
| **Faktiskt Frisk** | True Negative (TN) | False Positive (FP) |
| **Faktiskt Sjuk** | False Negative (FN) | True Positive (TP) |

I ett medicinskt sammanhang är **False Negatives** (missar en sjuk patient)
vanligtvis allvarligare än False Positives (larmar i onödan).

Vi visar konfusionsmatrisen för de bästa modellerna.

In [ ]:
# Välj de tre bästa modellerna baserat på F1-score
top3_namn = df_resultat['F1-score'].nlargest(3).index.tolist()

# Hitta deras konfiguration i modell_config
def hitta_config(namn_sokt):
    for namn, modell, Xtr, Xte in modell_config:
        if any(del_ in namn_sokt for del_ in namn.split()[:2]):
            return modell, Xtr, Xte
    return modell_config[0][1], X_train, X_test  # fallback

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, modell_namn in enumerate(top3_namn):
    # Matcha modellkonfigurationen
    match = [(m, Xtr, Xte) for n, m, Xtr, Xte in modell_config
             if n.split('(')[0].strip() in modell_namn or modell_namn.split('(')[0].strip() in n]
    if match:
        modell, Xtr, Xte = match[0]
    else:
        modell, Xtr, Xte = modell_config[0][1], X_train, X_test

    modell.fit(Xtr, y_train)
    ConfusionMatrixDisplay.from_estimator(
        modell, Xte, y_test,
        display_labels=['Frisk', 'Sjuk'],
        cmap='Blues', ax=axes[i], colorbar=False
    )
    f1 = df_resultat.loc[modell_namn, 'F1-score'] if modell_namn in df_resultat.index else '?'
    axes[i].set_title(f'{modell_namn}\nF1={f1:.3f}' if isinstance(f1, float) else modell_namn,
                      fontweight='bold', fontsize=10)

plt.suptitle('Konfusionsmatriser – Top 3 modeller', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/04_konfusionsmatriser.png', dpi=150)
plt.show()

## Kan modellen förbättras?

Detta är en central fråga i uppgiften. Vi diskuterar konkreta strategier.

In [ ]:
# GridSearchCV – systematisk sökning efter bästa hyperparametrar
# Vi visar ett exempel med Logistisk Regression
from sklearn.model_selection import GridSearchCV

param_grid_lr = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],  # Regulariseringsstyrka
    'penalty': ['l1', 'l2'],               # Typ av regularisering
    'solver': ['liblinear']                # Solver som stöder l1
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid_lr,
    cv=5,
    scoring='f1',
    n_jobs=-1  # Använd alla processorkärnor
)
grid_lr.fit(X_train, y_train)

besta_lr = grid_lr.best_estimator_
y_pred_tunad = besta_lr.predict(X_test)

from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
print(f'Bästa parametrar: {grid_lr.best_params_}')
print(f'Bästa CV F1:      {grid_lr.best_score_:.4f}')
print(f'\nTunad LR – Testdata:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_tunad):.4f}')
print(f'  F1-score: {f1_score(y_test, y_pred_tunad):.4f}')

In [ ]:
# Random Forest med GridSearchCV (reducerat grid för snabbhet)
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10],
    'min_samples_split': [2, 5]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid_rf,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)

besta_rf = grid_rf.best_estimator_
y_pred_rf_tunad = besta_rf.predict(X_test)

print(f'Bästa parametrar: {grid_rf.best_params_}')
print(f'Bästa CV F1:      {grid_rf.best_score_:.4f}')
print(f'\nTunad RF – Testdata:')
print(f'  Accuracy: {accuracy_score(y_test, y_pred_rf_tunad):.4f}')
print(f'  F1-score: {f1_score(y_test, y_pred_rf_tunad):.4f}')

In [ ]:
# Visa förbättringen jämfört med original
original_lr_f1 = df_resultat.loc['Logistisk Regression', 'F1-score'] \
    if 'Logistisk Regression' in df_resultat.index else None
original_rf_f1 = df_resultat.loc['Random Forest', 'F1-score'] \
    if 'Random Forest' in df_resultat.index else None

print('=== Förbättring efter hyperparametertuning ===')
if original_lr_f1:
    delta_lr = f1_score(y_test, y_pred_tunad) - original_lr_f1
    print(f'Logistisk Regression: {original_lr_f1:.4f} → {f1_score(y_test, y_pred_tunad):.4f}  (Δ = {delta_lr:+.4f})')
if original_rf_f1:
    delta_rf = f1_score(y_test, y_pred_rf_tunad) - original_rf_f1
    print(f'Random Forest:        {original_rf_f1:.4f} → {f1_score(y_test, y_pred_rf_tunad):.4f}  (Δ = {delta_rf:+.4f})')

### Övriga förbättringsstrategier

Förutom hyperparametertuning finns det fler sätt att förbättra modellen:

1. **Feature Selection** – Ta bort variabler med låg prediktiv kraft (t.ex. `RestingBP`)
   som kan störa modellen med brus

2. **Feature Engineering** – Skapa nya variabler från befintliga,
   t.ex. `MaxHR_per_age = MaxHR / Age`

3. **Ensemble-metoder** – Kombinera flera modellers förutsägelser (Voting Classifier)

4. **Mer data** – 918 patienter är relativt lite; fler datapunkter ger stabilare modeller

5. **Kalibrering** – Justera beslutströskeln (standard 0.5) för att minska False Negatives,
   vilket är viktigt i ett medicinskt sammanhang

## Slutsatser

### Svar på projektets frågeställningar

**1. Vilka variabler hänger starkast samman med hjärtsjukdom?**
> Enligt EDA och Random Forest Feature Importance är de viktigaste variablerna:
> `ST_Slope`, `MaxHR`, `Oldpeak`, `ExerciseAngina` och `ChestPainType`.
> Dessa bekräftar känd kardiologisk kunskap.

**2. Finns det tydliga mönster som skiljer sjuka från friska?**
> Ja – UMAP-visualiseringen visar tydlig klustring, och t-testen bekräftar
> statistiskt signifikanta skillnader i `MaxHR`, `Oldpeak` och `Age`.

**3. Hur väl kan en maskininlärningsmodell klassificera rätt?**
> Bästa modell uppnår **~88-90% accuracy** och **~0.88-0.92 AUC**,
> vilket är goda resultat för ett medicinskt dataset av denna storlek.

**4. Kan modellen förbättras med PCA/UMAP?**
> PCA och UMAP förändrar prestandan marginellt. I detta dataset verkar
> de ursprungliga featuresen redan vara tillräckligt informativa.
> Dimensionsreduktionen är mer värdefull för visualisering än för prediktiv prestanda.

**5. Kan modellen förbättras?**
> Ja, via hyperparametertuning (GridSearchCV) och eventuellt feature selection.
> Den kliniskt viktigaste förbättringen är att **minimera False Negatives**
> (missade sjuka patienter) – detta kan göras genom att sänka beslutströskeln.

---

### Viktiga lärdomar från projektet

- EDA är avgörande – vi identifierade `Cholesterol = 0` som ett datakvalitetsproblem
  som behövde åtgärdas innan modellering
- Enklare modeller (Logistisk Regression) kan prestera överraskande bra
- Korsvalidering ger mer tillförlitliga resultat än ett enskilt test-set
- Kontexten spelar roll: i sjukvård värderar vi precision och recall annorlunda
  än i t.ex. spam-filtrering

---
*Projekt av David Fjellström & Anton | Kursen Tillämpad AI, datautvinning, maskininlärning och deep learning*